<a href="https://colab.research.google.com/github/ompatil2004/Task6_Slashmark/blob/main/Task6_SlashMark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# 1. Load your dataset
df = pd.read_csv('songdata.csv')

# 2. Data Cleaning & Downsampling
# Take a sample of 5,000 songs to fit into standard memory
df_sample = df.sample(n=5000, random_state=42).reset_index(drop=True)

# Clean the lyrics (remove newline characters and make lowercase)
df_sample['text'] = df_sample['text'].str.replace('\n', ' ').str.lower()
# Optional: remove special characters
df_sample['text'] = df_sample['text'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x)))

print(f"Dataset shape after sampling: {df_sample.shape}")

# 3. Feature Extraction (NLP)
# We use TF-IDF to convert lyrics into a mathematical matrix of word importances
# stop_words='english' removes common words like 'the', 'is', 'and'
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

print("Analyzing lyrics and building the matrix... (This may take a minute)")
tfidf_matrix = tfidf.fit_transform(df_sample['text'])

# 4. Compute Similarity
# This calculates how similar every song's lyrics are to every other song
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Similarity matrix built successfully!")

# 5. Recommendation Function
def recommend_similar_songs(song_title, df, sim_matrix, top_n=5):
    # Check if the song exists in our sample
    if song_title not in df['song'].values:
        return f"Song '{song_title}' not found in the dataset sample."

    # Get the index of the song
    idx = df[df['song'] == song_title].index[0]

    # Get similarity scores for all songs compared to this one
    sim_scores = list(enumerate(sim_matrix[idx]))

    # Sort the songs based on similarity score (highest first)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar songs (skip the first one as it's the song itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the indices of these similar songs
    song_indices = [i[0] for i in sim_scores]

    # Return the recommendations
    print(f"\nBecause you listened to '{song_title}', you might like:")
    return df.iloc[song_indices][['song', 'artist']]

# 6. Test the Recommender!
# Let's pick a random song from our sample to test
test_song = df_sample['song'].iloc[10]
print(f"\nTesting with song: {test_song} by {df_sample['artist'].iloc[10]}")

recommendations = recommend_similar_songs(test_song, df_sample, cosine_sim)
display(recommendations)

Dataset shape after sampling: (5000, 4)
Analyzing lyrics and building the matrix... (This may take a minute)
Similarity matrix built successfully!

Testing with song: Blood In My Eyes by Marianne Faithfull

Because you listened to 'Blood In My Eyes', you might like:


,song,artist
3484,"Hey, Hey, Hey",Elvis Presley
693,Weird World,Backstreet Boys
4007,But The Blood,Kirk Franklin
4060,I Got You Babe,Etta James
941,Stackin' Paper,ZZ Top
